In [6]:
import pandas as pd
import os
from missforest import MissForest
import numpy as np

In [ ]:
# Iterate over complete simulated data directory and impute all datasets.
# Directory containing your pickle files
input_directory = 'simulated_data/'
output_directory = 'imputed_data/'
cat_variables_df = pd.read_csv('cat_variables.csv')
cat_variables = list(cat_variables_df['cat_var'])

# Iterate over all files in the directory
for filename in os.listdir(input_directory):
    if filename.endswith('.tsv'):
        # Full file path
        file_path = os.path.join(input_directory, filename)
        
        # Load simulated df.
        df = pd.read_csv(file_path, sep='\t', index_col=0)
        old_cols = list(df.columns)
        col_translator = {col : f'{x}' for x, col in enumerate(old_cols)}
        col_back_translator = {val : key for key, val in col_translator.items()}
        df.rename(columns=col_translator, inplace=True)
        
        # Map pandas NA to numpy NA.
        df = df.replace(pd.NA, np.nan)
        
        # Get filename without extension.
        name_without_ext = os.path.splitext(filename)[0]
        
        # Split by underscores.
        parts = name_without_ext.split('_')
                
        # Run imputer.
        cat_columns = [col_translator[old_col] for old_col in cat_variables]
        imputer = MissForest(categorical=cat_columns)
        imputed_df = imputer.fit_transform(df)
        imputed_df.rename(columns=col_back_translator, inplace=True)
        
        # Save imputed df.
        imputed_filename = os.path.join(output_directory, f'{name_without_ext}.tsv')
        imputed_df.to_csv(imputed_filename, sep='\t', index=True)